# K.Lab tiles computation

### Overview
Get the gdf of GADM and the desiferd tiles grid from PostgreSQL and the the intersected tiles as a list from the country to the coastal level.

### parameters
- db_name (str): Name of the database.
- user (str): Database username.
- password (str): Database password.
- host (str): Host address (e.g., "localhost").
- port (int): Database port (default PostGIS port is 5432).
- table_name (str): Name of the vector table to load.
- schema (str, optional): Database schema (default "public").
- geom_col (str, optional): Geometry column name (default "geom").

### Returns
- **df**: The function writes table with the country and the overlaped tiles.

### Author
- Rubén Crespo Ceballos


In [2]:
import geopandas as gpd
from sqlalchemy import create_engine, text

import pandas as pd
from shapely.ops import unary_union
import os
import gc

In [4]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    

def intersecting_tiles_list_fast(country_gdf, grid_gdf):
    """
    Vectorized version — uses geo-based spatial join + groupby.
    """
    # ensure CRS alignment
    if country_gdf.crs != grid_gdf.crs:
        print("There is a disalignment on the files.")
        grid_gdf = grid_gdf.to_crs(country_gdf.crs)

    # Get the geometry name
    geometry_name = country_gdf.geometry.name

    # spatial join (many-to-many)
    joined = grid_gdf.sjoin(country_gdf[['country_n', geometry_name]], how='inner', predicate='intersects')

    # group tile indices by country
    out_df = (joined.groupby('country_n')['tile_index']
                  .apply(list)
                  .reset_index()
                  .rename(columns={'country_n': 'country_name'}))

    return out_df


def intersecting_tiles_list(country_gdf, grid_gdf):
    """
    Returns a list of tile_index values from the grid that intersect each country polygon.

    Parameters:
    - country_gdf: GeoDataFrame with a 'country_n' column and country polygons
    - grid_gdf: GeoDataFrame with a 'tile_index' column and grid cell polygons

    Returns:
    - pandas DataFrame with columns ['country', 'tile_indices']
      where 'tile_indices' is a list of tile_index values intersecting the polygon.
    """
    results = []
    geom_col = country_gdf.geometry.name

    for idx, feature in country_gdf.iterrows():
        print(f"Working with country:{feature.get('country_n'), idx} from a total of {len(country_gdf)}" )
        
        geom = feature[geom_col]

        # Spatial filter: grid cells that intersect the geometry
        possible_tiles = grid_gdf[grid_gdf.intersects(geom)]

        # Extract intersecting tile indices
        tiles = possible_tiles["tile_index"].tolist()

        results.append({
            'country': feature.get('country_n'),
            'tile_indices': tiles
        })

    return pd.DataFrame(results)

def extract_country_coastlines(gadm_gdf, countries):
    """
    Extract true sea-facing coastlines for one or more countries
    from a GADM GeoDataFrame, excluding shared inland borders.

    Parameters
    ----------
    - gadm_gdf (geopandas.GeoDataFrame): GADM geometries with 'country_n' column for country names.
    - countries (str | list[str]): Country name or list of country names to extract coastlines for.

    Returns
    -------
    coast_gdf (geopandas.GeoDataFrame): GeoDataFrame containing only sea-facing coastlines 
    with a 'country_n' column for country names.
    """

    if isinstance(countries, str):
        countries = [countries]

    results = []
    counter = 1

    for country in countries:
        print(f"Working with country: {country}. Index {counter} out of {len(countries)}")
        counter += 1
        target = gadm_gdf[gadm_gdf["country_n"] == country]
        if target.empty:
            continue

        target_union = unary_union(target.geometry)

        # Only use countries that touch the target as neighbors
        gadm_gdf["touches_target"] = gadm_gdf.geometry.touches(target_union)
        neighbors = gadm_gdf[gadm_gdf["touches_target"] & (gadm_gdf["country_n"] != country)]

        # Merge neighboring polygons and subtract them from the boundary
        others_union = unary_union(neighbors.geometry)
        coastline_geom = target_union.boundary.difference(others_union)
        
        # Create a temporary GeoDataFrame
        coast_gdf = gpd.GeoDataFrame(
            {"country_n": [country]}, geometry=[coastline_geom], crs=gadm_gdf.crs
        ).explode(index_parts=False).reset_index(drop=True)

        # Clean geometries
        coast_gdf = coast_gdf[coast_gdf.is_valid & (coast_gdf.length > 0)]
        results.append(coast_gdf)

    if not results:
        raise ValueError(f"No valid coastlines extracted for {countries}")
    # Combine and dissolve by country
    combined = gpd.GeoDataFrame(pd.concat(results, ignore_index=True), crs=gadm_gdf.crs)
    dissolved = combined.dissolve(by="country_n", as_index=False)

    return dissolved

def extract_country_coastlines_fast(gadm_gdf, countries):
    """
    Optimized version of true sea-facing coastline extraction.
    Uses spatial index and avoids per-country full-table scans.
    """
    if isinstance(countries, str):
        countries = [countries]

    geom_col = gadm_gdf.geometry.name

    # Preselect only countries we need (+ neighbors)
    gadm = gadm_gdf[["country_n", geom_col]].copy()

    # Precompute unions for all countries once
    country_unions = (
        gadm.groupby("country_n")[geom_col]
            .apply(unary_union)
    )
    print("Precomputed all countries")
    results = []

    counter = 1

    # Build spatial index (auto in sjoin)
    for country in countries:
        print(f"Working with country: {country}. Index {counter} out of {len(countries)}")
        counter += 1
        
        if country not in country_unions:
            continue

        target_geom = country_unions[country]

        # Find *possible* touching neighbors via fast bounding-box join
        target_gdf = gpd.GeoDataFrame({"country_n": [country]}, geometry=[target_geom], crs=gadm.crs)

        # bounding-box spatial join → extremely fast
        hits = gpd.sjoin(gadm, target_gdf, predicate="intersects")

        # keep only OTHER countries that truly touch
        neighbors = hits[hits["country_n_left"] != country]
        neighbors = neighbors[neighbors.geometry.touches(target_geom)]

        # union neighbors only if there are any
        if len(neighbors) > 0:
            neighbors_union = unary_union(neighbors.geometry)
        else:
            neighbors_union = None

        # Compute sea-facing coastline
        coastline = (
            target_geom.boundary
            if neighbors_union is None
            else target_geom.boundary.difference(neighbors_union)
        )

        # Convert to GeoDataFrame
        coast_gdf = gpd.GeoDataFrame(
            {"country_n": [country]}, geometry=[coastline], crs=gadm.crs
        ).explode(index_parts=False)

        # filter out tiny/invalid pieces
        coast_gdf = coast_gdf[coast_gdf.is_valid & (coast_gdf.length > 0)]

        results.append(coast_gdf)

    combined = pd.concat(results, ignore_index=True)
    return combined.dissolve(by="country_n", as_index=False)



def filter_countries(gdf_gadm, countries):

    available_countries = gdf_gadm['country_n'].unique().tolist()
    selected_countries = countries.copy()

    # Check for missing countries
    missing_countries = [c for c in selected_countries if c not in available_countries]

    if missing_countries:
        return print("Warning: These countries are not in the DataFrame:", missing_countries)
    
    else:
        gdf_gadm_countries = gdf_gadm[gdf_gadm['country_n'].isin(countries)]
        return gdf_gadm_countries




In [5]:
"""Load the data"""
# Load the GADM
# gdf_gadm = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.100',
#     port=5555,
#     table_name='administrative_units_un_gadm_level0',
#     schema='public'
# )

# Load the grid
gdf_grid = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.233',
    port=5555,
    table_name='global_klab_1d_tiles',
    schema='klab_grids'
)

# gdf_coastline = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.100',
#     port=5555,
#     table_name='global_osm_coastline_2km_buffer_4326',
#     schema='public'
# )

# gdf_coastline = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.100',
#     port=5555,
#     table_name='global_osm_coastline_4326',
#     schema='public'
# )

Successfully loaded global_klab_1d_tiles (50760 features)


In [7]:
gdf_grid.head()

,gid,id,left,top,right,bottom,lon,lat,row,col,tile_index,geom
0,1,1.0,-180.0,84.0,-179.0,83.0,-179.5,83.5,0.0,0.0,62281.0,"MULTIPOLYGON (((-180.00000 84.00000, -179.0000..."
1,2,2.0,-180.0,83.0,-179.0,82.0,-179.5,82.5,1.0,0.0,61921.0,"MULTIPOLYGON (((-180.00000 83.00000, -179.0000..."
2,3,3.0,-180.0,82.0,-179.0,81.0,-179.5,81.5,2.0,0.0,61561.0,"MULTIPOLYGON (((-180.00000 82.00000, -179.0000..."
3,4,4.0,-180.0,81.0,-179.0,80.0,-179.5,80.5,3.0,0.0,61201.0,"MULTIPOLYGON (((-180.00000 81.00000, -179.0000..."
4,5,5.0,-180.0,80.0,-179.0,79.0,-179.5,79.5,4.0,0.0,60841.0,"MULTIPOLYGON (((-180.00000 80.00000, -179.0000..."


In [4]:
""" This is a temporal """
gdf_coastlines = gpd.read_file("coastline_by_1d_grid.shp")
# dissolved_gdf = gdf_gadm.dissolve()

# buffered = gdf_coastlines.buffer(0.027)

# gdf_coastlines_buffer = gpd.GeoDataFrame(
#     geometry=buffered,
#     crs="EPSG:4326"
# )

In [ ]:
gdf_coastlines.head()

In [5]:
coast = gdf_coastlines.copy()

# explode large multilines (VERY important)
coast = coast.explode(index_parts=False).reset_index(drop=True)

In [9]:
out_dir = r".\buffer_files"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
chunk_size = 500        # adjust to your RAM
buffer_dist = 0.027

n = len(coast)
print(f"Amount of items: {n}")

for start in range(0, n, chunk_size):
    end = min(start + chunk_size, n)
    outfile = os.path.join(out_dir, f"buffer_{start}_{end}.shp")

    # Skip if already processed (restart-safe)
    if os.path.exists(outfile):
        print(f"[SKIP] {outfile} already exists")
        continue

    print(f"[PROCESS] Geometries {start} → {end}")
    chunk = coast.iloc[start:end].copy()

    # Apply buffer with low resolution and fix geometry
    chunk["geometry"] = chunk.buffer(buffer_dist, resolution=1).buffer(0)

    # Save chunk immediately as Shapefile
    chunk.to_file(outfile, driver="ESRI Shapefile")
    print(f"[SAVED] Chunk saved as {outfile}")

    # Free memory
    del chunk
    gc.collect()

In [10]:
outno_holes_dir = r".\no_holes_buffer_files"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
print("Merging all chunk shapefiles...")
files = [os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith(".shp")]
files.sort()  # optional, ensures order

gdfs = []
for f in files:
    print(f"[READ] {f}")
    gdf = gpd.read_file(f)
    gdfs.append(gdf)

merged = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
print(f"Total geometries after merge: {len(merged)}")

# Optional: dissolve all into single geometry
print("Dissolving merged geometries into one...")
merged_dissolved = merged.dissolve()
print("Dissolve complete.")

# -------------------------------
# SAVE FINAL RESULT
# -------------------------------
final_outfile = "coast_buffer_final.shp"
merged_dissolved.to_file(final_outfile, driver="ESRI Shapefile")
print(f"All done! Final buffered coastline saved as '{final_outfile}'")

In [10]:
gdf_coastlines = gdf_coastlines.drop("tile_index", axis=1)

In [ ]:
gdf_coastlines["geometry"] = gdf_coastlines.buffer(0.027)

C:\Users\admin\AppData\Local\Temp\ipykernel_10732\1402808191.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_coastlines["geometry"] = gdf_coastlines.buffer(0.027)


In [ ]:
dissolved_gadm = gdf_gadm.dissolve()
lines_coast = dissolved_gadm.boundary

coast_gdf = gpd.GeoDataFrame(geometry=lines_coast)
coast_gdf = coast_gdf.explode(index_parts=False)
coast_gdf = coast_gdf.reset_index(drop=True)

coast_gdf.to_file("gadm_coast.shp")

In [ ]:
# Optional but VERY important for speed
# (Shapely 2 / PyGEOS backend)
coast = gdf_coastlines.copy()
grid = gdf_grid.copy()

# Make geometries valid (prevents topology errors & slowdowns)
coast["geometry"] = coast.make_valid()
grid["geom"] = grid.make_valid()

# Keep only needed columns
grid = grid[["tile_index", "geom"]]   # cell_id = your grid unique ID

# ---- SPLIT coastline by grid (spatial-index accelerated) ----
split = gpd.overlay(
    coast,
    grid,
    how="intersection",
    keep_geom_type=True
)

# ---- DISSOLVE per grid cell ----
result = split.dissolve(
    by="tile_index",
    as_index=False
)

# Save result
result.to_file("coastline_by_1d_grid.shp", driver="ESRI Shapefile")

In [ ]:
"""Country based tile listing"""
# Select the country
countries = ["Colombia", "Ecuador", "Peru"]
gdf_gadm_countries = filter_countries(gdf_gadm, countries)

# Get the tiles
# df_result = intersecting_tiles_list(gdf_gadm_countries, gdf_grid)

In [ ]:
"""Direct tile listing"""
# Get all the tiles
df_result = intersecting_tiles_list_fast(gdf_gadm, gdf_grid)

In [ ]:
"""Merge duplicates and add an aditional column with the tiles formated """
# Merge duplicates by extending lists
df_merged = df_result.groupby('country_name', as_index=False).agg({'tile_index': sum})

# Convert float list → int list → string
df_merged['List_values'] = df_merged['tile_index'].apply(lambda lst: ', '.join(str(int(v)) for v in lst))

In [ ]:
"""Coastal based tile listing"""
# Selected countries
# countries = ["Kenya", "Somalia"] # Example

# All countries
countries = gdf_gadm["country_n"].tolist()

country_coastlines = extract_country_coastlines(gdf_gadm, countries)


In [24]:
country_coastlines.head()

,country_n,geometry
0,Afghanistan,"LINESTRING (73.65508 36.88848, 73.64670 36.88802)"
1,Albania,"MULTILINESTRING ((19.99213 39.77490, 19.99181 ..."
2,Algeria,"MULTILINESTRING ((-2.21190 35.08430, -2.21151 ..."
3,American Samoa,"MULTILINESTRING ((-168.14333 -14.54667, -168.1..."
4,Angola,"MULTILINESTRING ((11.73597 -16.67291, 11.73597..."


In [25]:
"""Compute the tiles"""
df_result = intersecting_tiles_list_fast(country_coastlines, gdf_grid)

In [26]:
"""Export the results"""
df_result.to_csv("1d_tiles_coast_countries_results_merged.csv", index=False)